In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)
print(len(documents))

72


In [3]:
print(documents[0].keys())

dict_keys(['content', 'filename'])


In [2]:
from minsearch import Index
def build_index(documents):
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index
# index = build_index(documents)

In [10]:
def search_files(index, query, filename_filter=None, num_results=5):
    # Basic search
    results = index.search(
        query=query,
        filter_dict={'filename': filename_filter} if filename_filter else {},
        num_results=num_results,
        boost_dict={'content': 1.0}
    )
    return results

In [8]:
query = "How does the agentic loop keep calling the model until it stops?"
results = search_files(index, query)
for x in results:
    print(f"Filename: {x['filename']}")
    print(f"Content: {x['content'][:200]}...")  # Print the first 200 characters
    

Filename: 01-agentic-rag/lessons/14-agentic-loop.md
Content: # The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we did function calling by hand. We sent a
...
Filename: 01-agentic-rag/lessons/15-frameworks.md
Content: # ToyAIKit

Video: [Watch this lesson](https://www.youtube.com/watch?v=PQpQOR3Un3w&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

The handwritten agent loop from the previous lesson is educational but
repe...
Filename: 01-agentic-rag/lessons/13-function-calling.md
Content: # Function Calling

Video: [Watch this lesson](https://www.youtube.com/watch?v=CeEki_0mdGo&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson we built a RAG pipeline with `RAGBase.rag()`...
Filename: 01-agentic-rag/lessons/11-agents-intro.md
Content: # Agents

Video: [Watch this lesson](https://www.youtube.com/watch?v=6uG4_Ivv60E&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In Part 1 of this m

In [3]:
from dotenv import load_dotenv
load_dotenv()

from rag_helper import RAGBaseHomework
from openai import OpenAI

index = build_index(documents)

openai_client = OpenAI()

assistant = RAGBaseHomework(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

Input tokens: 7136
The loop keeps calling the model by checking whether the model returned any `function_call` items.

- It sends the current `messages` to `responses.create(...)`
- It looks through `response.output`
- If it finds a `function_call`, it runs the tool, appends the tool result to `messages`, and sets `has_function_calls = True`
- If there are no function calls, it breaks out of the `while True` loop

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In short: it keeps looping until the model returns a final message with no more tool calls.


In [9]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))
print(chunks[0].keys())

295
dict_keys(['start', 'content', 'filename'])


In [11]:
index_cnk = build_index(chunks)
from dotenv import load_dotenv
load_dotenv()

from rag_helper import RAGBaseHomework
from openai import OpenAI

index = build_index(documents)

openai_client = OpenAI()

assistant = RAGBaseHomework(
    index=index_cnk,
    llm_client=openai_client,
)

answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

Input tokens: 2319
The loop keeps calling the model with a `while True`, then checks whether any `function_call` items came back.

- If there is at least one function call, the code runs the tool, appends the tool result to `messages`, and loops again.
- If there are no function calls in that turn, `has_function_calls` stays `False`, and the loop breaks.

So the stop condition is: **no function calls this turn means the model has finished and the loop ends.**


In [5]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
"""

question = 'How does the agentic loop work, and how is it different from plain RAG?'

In [3]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
index_cnk = build_index(chunks)

In [4]:
# def search(query: str) -> dict[str, str]:
#     """
#     Search the FAQ database for entries matching the given query.
#     """
#     return index.search(
#         query,
#         num_results=5,
#         boost_dict={'question': 3.0, 'section': 0.5},
#         filter_dict={'course': 'llm-zoomcamp'}
    # )
def search_files(query, filename_filter=None, num_results=5):
    # Basic search
    results = index_cnk.search(
        query=query,
        filter_dict={'filename': filename_filter} if filename_filter else {},
        num_results=num_results,
        boost_dict={'content': 1.0}
    )
    return results

In [10]:
search_tool = {
    "type": "function",
    'name': 'search_files',
    'description': 'Search the course materials for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course materials.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [7]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [11]:
agent_tools = Tools()
agent_tools.add_tool(search_files, search_tool)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search_files',
  'description': 'Search the course materials for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course materials.'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [12]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [13]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)
result = runner.loop(
    prompt=question,
    callback=callback,
)
result

-> Response received


-> Response received


LoopResult(new_messages=[EasyInputMessage(content="\nYou're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.\n", role='developer', phase=None, type=None), EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"agentic loop RAG"}', call_id='call_qFehWSCFdtcUl3E7mHIrIHtK', name='search_files', type='function_call', id='fc_0db3ebc522829114006a38230bc15481a3891b1627883817ef', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"plain RAG retrieval augmented generation"}', call_id='call_ajzDU5mXlxBgHfcSmfctgLgZ', name='search_files', type='function_call', id='fc_0db3ebc522829114006a38230bc16881a392764b99a86c82af', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"agentic workflow loop retrieve act observe"}